# Customer Segmentation + Pricing Strategy
**Dataset:** UCI Online Retail II — 1,067,371 real transactions (2009–2011)  
**Pipeline:** Data cleaning → RFM features → KMeans / DBSCAN / GMM → Churn risk → WTP model → Pricing strategy

Run cells top-to-bottom. On first run, the UCI dataset downloads automatically (~30 seconds).

In [1]:
!pip install numpy pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


In [2]:
!pip install ucimlrepo

## 1. Load + Clean Data

In [10]:
!pip install certifi
!/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m certifi

/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14: No module named certifi


In [11]:
import os
os.system('/Applications/Python\\ 3.14/Install\\ Certificates.command')

 -- pip install --upgrade certifi
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
 -- removing any existing file or link
 -- creating symlink to certifi certificate bundle
 -- setting permissions
 -- update complete



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


0

In [3]:
import os
print(os.getcwd())
print(os.listdir())

/Users/padmapriyavenkatesh/Downloads/retail-segmentation/Customer-Segmentation-Pricing-Strategy/retail-segmentation/notebooks
['full_analysis.ipynb']


In [4]:
import os
# Check what's in the parent folders
print(os.listdir('..'))         # one level up
print(os.listdir('../..'))      # two levels up

['.DS_Store', 'requirements.txt', 'Makefile', 'tests', 'docs', 'README.md', 'setup.py', '.gitignore', '.venv', 'venv', 'main.py', 'data', 'notebooks', 'src']
['.DS_Store', 'retail-segmentation', 'README.md', 'retail-segmentation 2.zip', '.git']


In [6]:
with open('../data/load_data.py', 'r') as f:
    print(f.read())

"""
data/load_data.py
Downloads the UCI Online Retail II dataset (~1M rows, 2009-2011)
directly from the UCI ML Repository and saves it as a CSV.

Usage:
    python data/load_data.py

No Kaggle account needed. The ucimlrepo library pulls it directly.
"""

from pathlib import Path
from ucimlrepo import fetch_ucirepo
import pandas as pd


def download_raw_data(out_path: str = "data/raw_retail.csv") -> pd.DataFrame:
    """
    Fetch the UCI Online Retail II dataset (id=502) and save to CSV.

    Returns:
        Raw DataFrame with 8 columns and ~1M rows.
    """
    print("Downloading UCI Online Retail II...")
    print("(~1,067,000 rows — this takes about 30 seconds)\n")

    retail = fetch_ucirepo(id=502)
    df = retail.data.features

    # Combine with targets if they exist
    if retail.data.targets is not None:
        targets = retail.data.targets
        if not targets.empty:
            df = pd.concat([df, targets], axis=1)

    print(f"  Shape        : {df.shape[0]:,} rows x {d

In [7]:
import os

content = """from pathlib import Path
import urllib.request
import zipfile
import pandas as pd

def download_raw_data(out_path: str = "../data/raw_retail.csv") -> pd.DataFrame:
    out_path = Path(out_path)
    out_path.parent.mkdir(exist_ok=True)

    zip_path = out_path.parent / "online_retail_ii.zip"
    xlsx_path = out_path.parent / "online_retail_II.xlsx"

    if not xlsx_path.exists():
        print("Downloading UCI Online Retail II...")
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as z:
            print("Files in zip:", z.namelist())
            z.extractall(out_path.parent)
        print("Download complete!")

    print("Reading Excel file...")
    df1 = pd.read_excel(xlsx_path, sheet_name="Year 2009-2010")
    df2 = pd.read_excel(xlsx_path, sheet_name="Year 2010-2011")
    df = pd.concat([df1, df2], ignore_index=True)

    print(f"  Shape     : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"  Columns   : {list(df.columns)}")

    df.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")
    return df
"""

path = '../data/load_data.py'
with open(path, 'w') as f:
    f.write(content)

# Verify
with open(path, 'r') as f:
    print(f.read())

from pathlib import Path
import urllib.request
import zipfile
import pandas as pd

def download_raw_data(out_path: str = "../data/raw_retail.csv") -> pd.DataFrame:
    out_path = Path(out_path)
    out_path.parent.mkdir(exist_ok=True)

    zip_path = out_path.parent / "online_retail_ii.zip"
    xlsx_path = out_path.parent / "online_retail_II.xlsx"

    if not xlsx_path.exists():
        print("Downloading UCI Online Retail II...")
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as z:
            print("Files in zip:", z.namelist())
            z.extractall(out_path.parent)
        print("Download complete!")

    print("Reading Excel file...")
    df1 = pd.read_excel(xlsx_path, sheet_name="Year 2009-2010")
    df2 = pd.read_excel(xlsx_path, sheet_name="Year 2010-2011")
    df = pd.concat([df1, df2], ignore_index=True)

    print(f"  Shape     : {df.shape[0]:,

In [9]:
!pip install certifi
!/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m certifi

/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14: No module named certifi


In [12]:
import sys
sys.path.insert(0, '..')

# Clear cache
for key in list(sys.modules.keys()):
    if 'load_data' in key or 'data' in key:
        del sys.modules[key]

from data.load_data import download_raw_data
df_raw = download_raw_data()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

In [13]:
import sys
sys.path.insert(0, '..')

from data.load_data import download_raw_data
df_raw = download_raw_data()

(~1,067,000 rows — this takes about 30 seconds)



DatasetNotFoundError: "Online Retail II" dataset (id=502) exists in the repository, but is not available for import. Please select a dataset from this list: https://archive.ics.uci.edu/datasets?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true

In [7]:
from data.load_data import download_raw_data
df_raw = download_raw_data()

ModuleNotFoundError: No module named 'data'

In [5]:
from data.clean_data import clean
df_clean = clean()
df_clean.head()

ModuleNotFoundError: No module named 'data'

In [ ]:
print('Rows after cleaning:', f'{len(df_clean):,}')
print('Customers:', f"{df_clean['customer_id'].nunique():,}")
print('Date range:', df_clean['invoice_date'].min().date(), '→', df_clean['invoice_date'].max().date())
print('Total revenue: £', f"{df_clean['revenue'].sum():,.2f}")
print('Countries:', df_clean['country'].nunique())

## 2. Feature Engineering — RFM

In [ ]:
from src.features import build_customer_features, scale_features, CLUSTER_FEATURES

customers = build_customer_features(df_clean)
customers.describe().round(2)

In [ ]:
# Distribution of RFM scores
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes,
                           ['recency_days', 'frequency', 'monetary'],
                           ['#4f8ef7', '#2dd4bf', '#f59e0b']):
    ax.hist(customers[col].clip(upper=customers[col].quantile(0.99)),
            bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Customers')
plt.suptitle('RFM Distributions (clipped at 99th percentile)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# RFM label distribution
customers['rfm_label'].value_counts()

## 3. Clustering — Elbow + BIC Model Selection

In [ ]:
from src.clustering import elbow_curve, gmm_bic_search

X, scaler = scale_features(customers, CLUSTER_FEATURES)
print('Feature matrix:', X.shape)

In [ ]:
print('KMeans elbow analysis:')
elbow_df = elbow_curve(X, k_range=range(2, 9))

In [ ]:
# Plot elbow
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(elbow_df['k'], elbow_df['inertia'], 'o-', color='#4f8ef7', lw=2, label='Inertia')
ax2.plot(elbow_df['k'], elbow_df['silhouette'], 's--', color='#2dd4bf', lw=2, label='Silhouette')
ax1.axvline(4, color='#f59e0b', linestyle=':', lw=1.5, label='k=4 selected')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')
ax2.set_ylabel('Silhouette')
lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, l1 + l2, frameon=False)
plt.title('KMeans Elbow Curve')
plt.tight_layout(); plt.show()

In [ ]:
print('GMM BIC search:')
bic_df = gmm_bic_search(X, n_range=range(2, 7))

## 4. Run All Three Clustering Models

In [ ]:
from src.clustering import run_all_models, comparison_table

results = run_all_models(X, k=4)

print('\nModel comparison:')
comparison_table(results)

In [ ]:
# PCA visualisation
from sklearn.decomposition import PCA

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
labels = results['gmm']['labels']
colors = ['#4f8ef7', '#2dd4bf', '#f97316', '#f59e0b']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (model_name, color_labels) in zip(axes, [
    ('GMM', results['gmm']['labels']),
    ('KMeans', results['kmeans']['labels'])
]):
    for i in range(4):
        mask = color_labels == i
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=5, alpha=0.3, color=colors[i], label=f'Cluster {i}', rasterized=True)
    ax.set_title(f'{model_name} — PCA Projection')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.legend(frameon=False, fontsize=8, markerscale=3)

plt.tight_layout(); plt.show()

## 5. Label Segments + Churn Risk

In [ ]:
from src.clustering import label_segments
from src.churn import compute_churn_risk, churn_summary

customers = label_segments(
    customers,
    results['gmm']['labels'],
    results['gmm']['proba']
)
customers['churn_risk'] = compute_churn_risk(customers)

churn_summary(customers)

In [ ]:
# Churn risk by segment
seg_colors = {
    'Champions': '#4f8ef7', 'Loyal Customers': '#2dd4bf',
    'At Risk': '#f97316', 'Needs Attention': '#f59e0b'
}
fig, ax = plt.subplots(figsize=(10, 4))
for seg, grp in customers.groupby('segment_name'):
    ax.hist(grp['churn_risk'], bins=30, alpha=0.55,
            color=seg_colors.get(seg, '#888'), label=seg, density=True)
ax.axvline(0.65, color='#ef4444', lw=1.5, linestyle='--', label='High-risk threshold')
ax.set_xlabel('Churn Risk Score'); ax.set_ylabel('Density')
ax.set_title('Churn Risk Distribution by Segment')
ax.legend(frameon=False); plt.tight_layout(); plt.show()

## 6. WTP Model + SHAP

In [ ]:
from src.wtp_model import build_wtp_target, train_wtp_model, predict_wtp

customers['wtp_proxy'] = build_wtp_target(customers)
wtp_model, wtp_features, metrics = train_wtp_model(customers)
customers['wtp_predicted'] = predict_wtp(wtp_model, customers, wtp_features)

print('\nMetrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
import shap
from src.wtp_model import compute_shap, shap_importance

shap_vals, X_sample = compute_shap(wtp_model, customers, wtp_features, sample_n=500)
shap.summary_plot(shap_vals, X_sample, show=True)

In [ ]:
# Feature importance table
shap_importance(shap_vals, X_sample).head(10)

## 7. Pricing Strategy

In [ ]:
from src.pricing import build_pricing_strategy, print_strategy

strategy = build_pricing_strategy(customers)
print_strategy(strategy)
strategy

In [ ]:
# MRR uplift chart
tier_colors = {'Enterprise':'#4f8ef7','Pro':'#2dd4bf','Win-Back':'#f97316','Starter':'#f59e0b'}
fig, ax = plt.subplots(figsize=(10, 4))
colors = [tier_colors.get(t, '#888') for t in strategy['recommended_tier']]
ax.barh(strategy['segment_name'], strategy['est_mrr_uplift'], color=colors, alpha=0.85)
ax.set_xlabel('Estimated MRR Uplift (£)')
ax.set_title('Pricing Strategy — Projected MRR Uplift by Segment')
for i, (v, row) in enumerate(zip(strategy['est_mrr_uplift'], strategy.itertuples())):
    ax.text(v + strategy['est_mrr_uplift'].max()*0.01, i,
            f"£{v:,} ({row.est_conversion_pct:.0f}% conv.)",
            va='center', fontsize=10)
ax.set_xlim(right=strategy['est_mrr_uplift'].max() * 1.5)
plt.tight_layout(); plt.show()

## 8. Segment Summary Table

In [ ]:
summary = customers.groupby('segment_name').agg(
    n                 = ('customer_id',    'count'),
    avg_recency       = ('recency_days',   'mean'),
    avg_frequency     = ('frequency',      'mean'),
    avg_monetary      = ('monetary',       'mean'),
    avg_churn_risk    = ('churn_risk',     'mean'),
    avg_wtp           = ('wtp_predicted',  'mean'),
).round(1)

summary['pct_of_base'] = (summary['n'] / summary['n'].sum() * 100).round(1)
summary.sort_values('avg_monetary', ascending=False)